# Figure 7 — Interactive Heat Map of Ghana Urban Centres

This cell adds an interactive Folium map to the project. Each city marker shows:
- Mean temperature (1990 vs 2023)
- Temperature anomaly
- Heatwave days (1990 vs 2023)
- Rainfall change
- Projected 2040 temperature

The map is also saved as a standalone HTML file in `outputs/`.

In [1]:
# ── Cell 11: Install folium if needed ────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'folium', '-q'])
print('folium ready.')

folium ready.


In [2]:
# ── Cell 12: Interactive Folium Map ──────────────────────────────────────────
import folium
import pandas as pd
from scipy import stats
import numpy as np

# Load data
df = pd.read_csv('../data/ghana_urban_heatwave.csv')

# City coordinates
coords = {
    'Accra':  (5.6037,  -0.1870),
    'Kumasi': (6.6885,  -1.6244),
    'Tamale': (9.4075,  -0.8533)
}

# City region labels
regions = {
    'Accra':  'Greater Accra Region — Coastal South',
    'Kumasi': 'Ashanti Region — Forest Zone',
    'Tamale': 'Northern Region — Guinea Savanna'
}

# Marker colours matching project palette
marker_colors = {
    'Accra':  '#E63946',
    'Kumasi': '#2A9D8F',
    'Tamale': '#E9C46A'
}

# ── Build map ────────────────────────────────────────────────────────────────
m = folium.Map(
    location=[7.9, -1.0],
    zoom_start=7,
    tiles='CartoDB positron',
    control_scale=True
)

# Add a dark tile layer option
folium.TileLayer('CartoDB dark_matter', name='Dark Mode').add_to(m)
folium.TileLayer('OpenStreetMap', name='Street Map').add_to(m)

for city in ['Accra', 'Kumasi', 'Tamale']:
    grp  = df[df['city'] == city].sort_values('year')
    lat, lon = coords[city]

    # Stats
    temp_1990  = grp[grp['year'] == 1990]['mean_temp_c'].values[0]
    temp_2023  = grp[grp['year'] == 2023]['mean_temp_c'].values[0]
    anomaly    = grp[grp['year'] == 2023]['anomaly_c'].values[0]
    hw_1990    = int(grp[grp['year'] == 1990]['heatwave_days'].values[0])
    hw_2023    = int(grp[grp['year'] == 2023]['heatwave_days'].values[0])
    rain_1990  = int(grp[grp['year'] == 1990]['rainfall_mm'].values[0])
    rain_2023  = int(grp[grp['year'] == 2023]['rainfall_mm'].values[0])
    pop_2023   = grp[grp['year'] == 2023]['urban_pop_millions'].values[0]

    # Projected 2040
    slope, intercept, _, _, _ = stats.linregress(grp['year'], grp['mean_temp_c'])
    proj_2040 = round(slope * 2040 + intercept, 1)

    # Bubble radius scaled to heatwave days
    radius = hw_2023 * 350

    # ── Heatwave bubble (background circle) ──────────────────────────────────
    folium.Circle(
        location=[lat, lon],
        radius=radius,
        color=marker_colors[city],
        fill=True,
        fill_color=marker_colors[city],
        fill_opacity=0.18,
        weight=2,
        tooltip=f'{city} — bubble size = {hw_2023} heatwave days in 2023'
    ).add_to(m)

    # ── Popup HTML ────────────────────────────────────────────────────────────
    popup_html = f"""
    <div style="font-family: Arial, sans-serif; width: 260px; padding: 4px;">
        <div style="background:{marker_colors[city]}; color:white; padding:8px 12px;
                    border-radius:6px 6px 0 0; margin:-4px -4px 8px -4px;">
            <b style="font-size:16px;">{city}</b><br>
            <span style="font-size:11px; opacity:0.9;">{regions[city]}</span>
        </div>

        <table style="width:100%; font-size:12px; border-collapse:collapse;">
            <tr style="background:#f8f8f8;">
                <td style="padding:5px 8px; color:#555;">🌡️ Mean Temp 1990</td>
                <td style="padding:5px 8px; font-weight:bold;">{temp_1990}°C</td>
            </tr>
            <tr>
                <td style="padding:5px 8px; color:#555;">🌡️ Mean Temp 2023</td>
                <td style="padding:5px 8px; font-weight:bold; color:#c0392b;">{temp_2023}°C</td>
            </tr>
            <tr style="background:#f8f8f8;">
                <td style="padding:5px 8px; color:#555;">📈 Temp Anomaly</td>
                <td style="padding:5px 8px; font-weight:bold; color:#c0392b;">+{anomaly}°C</td>
            </tr>
            <tr>
                <td style="padding:5px 8px; color:#555;">☀️ Heatwave Days 1990</td>
                <td style="padding:5px 8px; font-weight:bold;">{hw_1990} days/yr</td>
            </tr>
            <tr style="background:#f8f8f8;">
                <td style="padding:5px 8px; color:#555;">☀️ Heatwave Days 2023</td>
                <td style="padding:5px 8px; font-weight:bold; color:#c0392b;">{hw_2023} days/yr</td>
            </tr>
            <tr>
                <td style="padding:5px 8px; color:#555;">🌧️ Rainfall 1990</td>
                <td style="padding:5px 8px; font-weight:bold;">{rain_1990} mm</td>
            </tr>
            <tr style="background:#f8f8f8;">
                <td style="padding:5px 8px; color:#555;">🌧️ Rainfall 2023</td>
                <td style="padding:5px 8px; font-weight:bold; color:#2980b9;">{rain_2023} mm</td>
            </tr>
            <tr>
                <td style="padding:5px 8px; color:#555;">👥 Urban Pop 2023</td>
                <td style="padding:5px 8px; font-weight:bold;">{pop_2023:.2f}M</td>
            </tr>
            <tr style="background:#fff3cd;">
                <td style="padding:5px 8px; color:#555;">🔮 Projected 2040</td>
                <td style="padding:5px 8px; font-weight:bold; color:#e67e22;">{proj_2040}°C</td>
            </tr>
        </table>
        <div style="margin-top:8px; font-size:10px; color:#888; text-align:center;">
            Source: ERA5 / NASA POWER / GMet
        </div>
    </div>
    """

    # ── City marker pin ───────────────────────────────────────────────────────
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=f'<b>{city}</b> — click for full data',
        icon=folium.Icon(color='red' if city=='Accra' else
                              'green' if city=='Kumasi' else 'orange',
                         icon='thermometer-half', prefix='fa')
    ).add_to(m)

# ── Legend ────────────────────────────────────────────────────────────────────
legend_html = """
<div style="position: fixed; bottom: 40px; left: 40px; z-index: 1000;
            background: white; padding: 14px 18px; border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.2); font-family: Arial; font-size: 12px;">
    <b style="font-size:13px;">🌡️ Urban Heat Wave Trends</b><br>
    <span style="font-size:11px; color:#888;">Ghana — 1990 to 2023</span><br><br>
    <span style="color:#E63946;">●</span> <b>Accra</b> — Coastal South<br>
    <span style="color:#2A9D8F;">●</span> <b>Kumasi</b> — Ashanti Forest<br>
    <span style="color:#E9C46A;">●</span> <b>Tamale</b> — Northern Savanna<br><br>
    <span style="font-size:11px; color:#888;">Bubble size = heatwave days (2023)<br>
    Click markers for full data</span>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# ── Title ─────────────────────────────────────────────────────────────────────
title_html = """
<div style="position: fixed; top: 12px; left: 50%; transform: translateX(-50%);
            z-index: 1000; background: white; padding: 10px 20px;
            border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.2);
            font-family: Arial; font-size: 14px; font-weight: bold; color: #333;">
    Urban Heat Wave Trends — Ghana (1990–2023)
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

# ── Layer control ─────────────────────────────────────────────────────────────
folium.LayerControl().add_to(m)

# ── Save HTML ─────────────────────────────────────────────────────────────────
map_path = '../outputs/ghana_heatwave_map.html'
m.save(map_path)
print(f'Interactive map saved to: {map_path}')
print('Open ghana_heatwave_map.html in your browser for the full interactive version.')

# Display inline in notebook
m

Interactive map saved to: ../outputs/ghana_heatwave_map.html
Open ghana_heatwave_map.html in your browser for the full interactive version.
